# Bayesian Spatial Diagnostics

This notebook is the canonical user guide for Bayesian Lagrange-Multiplier (LM)
specification testing in `bayespecon`.  It covers

1. the **inventory of LM tests** exposed by the package (cross-section, panel,
   and spatial-flow families),
2. **direct calls** to the test functions for cross-sectional models,
3. the **method-based API** (`spatial_diagnostics()` and
   `spatial_diagnostics_decision()`) on every fitted model,
4. a **DGP-recovery stress test** that simulates each cross-sectional DGP and
   checks whether the decision tree lands on the correct generating model, and
5. short worked examples for **panel** and **spatial-flow** diagnostics.

## Background

The classical Anselin–Florax / Koley–Bera LM family tests:

| Test | $H_0$ | Alternative | df |
|------|-------|-------------|-----|
| LM-Lag | $\rho = 0$ | SAR | 1 |
| LM-Error | $\lambda = 0$ | SEM | 1 |
| LM-WX | $\gamma = 0$ | SLX | $k_{wx}$ |
| LM-SDM (joint) | $\rho = \gamma = 0$ | SDM | $1 + k_{wx}$ |
| LM-SLX-Error (joint) | $\lambda = \gamma = 0$ | SDEM | $1 + k_{wx}$ |
| Robust LM-Lag | $\rho = 0$ robust to $\lambda$ | SAR vs SEM | 1 |
| Robust LM-Error | $\lambda = 0$ robust to $\rho$ | SEM vs SAR | 1 |
| Robust LM-Lag-SDM | $\rho = 0$ robust to $\gamma$ | SDM | 1 |
| Robust LM-WX | $\gamma = 0$ robust to $\rho$ | SDM | $k_{wx}$ |
| Robust LM-Error-SDEM | $\lambda = 0$ robust to $\gamma$ | SDEM | 1 |
| LM-WX-SEM | $\gamma = 0$ in SEM | SDEM | $k_{wx}$ |
| LM-Error-SDM | $\lambda = 0$ in SDM | SDARAR | 1 |
| LM-Lag-SDEM | $\rho = 0$ in SDEM | SDARAR | 1 |

The Bayesian analogue evaluates the LM statistic at every posterior draw,
yielding a **posterior distribution** of the LM statistic rather than a single
point estimate.  Robust variants use the **Neyman orthogonal score** of
Doğan, Taşpınar & Bera (2021) to remove correlation between the test score and
the nuisance score:

$$g_\psi^* = g_\psi - J_{\psi\phi \cdot \sigma}\,J_{\phi\phi\cdot\sigma}^{-1}\,g_\phi.$$

The same machinery extends to balanced panels (with a $T$ multiplier on the
information matrix) and to origin–destination flow models built on Kronecker
weight matrices $W_d, W_o, W_w$.  All three families are demonstrated below.

In [ ]:
import warnings

import libpysal
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from spreg import OLS as SpregOLS
from spreg import LMtests as SpregLMtests

from bayespecon import (
    OLS,
    SAR,
    SDEM,
    SDM,
    SEM,
    SLX,
    bayesian_lm_error_sdm_test,
    bayesian_lm_error_test,
    bayesian_lm_lag_sdem_test,
    bayesian_lm_lag_test,
    bayesian_lm_sdm_joint_test,
    bayesian_lm_slx_error_joint_test,
    bayesian_lm_wx_sem_test,
    bayesian_lm_wx_test,
    bayesian_robust_lm_error_sar_test,
    bayesian_robust_lm_error_sdem_test,
    bayesian_robust_lm_error_sdm_test,
    bayesian_robust_lm_error_test,
    bayesian_robust_lm_lag_sdem_test,
    bayesian_robust_lm_lag_sdm_test,
    bayesian_robust_lm_lag_sem_test,
    bayesian_robust_lm_lag_test,
    bayesian_robust_lm_wx_sem_test,
    bayesian_robust_lm_wx_test,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## 1. Generate Spatial Data

We create a simple spatial DGP with known parameters to validate the tests. Under the null hypothesis (no spatial effects), the Bayesian LM statistics should be small with high p-values.

In [ ]:
import geopandas as gpd
from libpysal.graph import Graph
from libpysal.weights import Rook

# Generate data under H0: no spatial effects
np.random.seed(42)

# Load Columbus dataset for spatial weights
columbus_path = libpysal.examples.get_path("columbus.shp")
gdf = gpd.read_file(columbus_path)

# Create a Graph (modern libpysal API) for bayespecon models
# Row-standardize the graph so spatial models work correctly
g = Graph.build_contiguity(gdf, rook=True).transform("r")
n = g.n

# Legacy W for spreg comparison
w_spreg = Rook.from_shapefile(columbus_path)
w_spreg.transform = "r"

# Get sparse and dense W matrices
W_sparse = g.sparse.tocsr().astype(np.float64)
W_dense = np.array(W_sparse.todense())

# Design matrix
k = 3
X = np.column_stack([np.ones(n), np.random.normal(size=(n, k - 1))])
beta_true = np.array([1.0, 2.0, -1.5])
y = X @ beta_true + np.random.normal(scale=1.0, size=n)

print(f"W shape: {W_sparse.shape}, nnz: {W_sparse.nnz}")

## 2. Fit Null Models

The Bayesian LM tests require posterior draws from the **null model** (the model under H₀). Different tests use different null models:

- **LM-WX test**: SAR model (includes ρ but not γ)
- **LM-SDM joint test**: OLS model (no spatial params)
- **LM-SLX-Error joint test**: OLS model (no spatial params)
- **Robust LM-Lag-SDM**: SLX model (includes γ but not ρ)
- **Robust LM-WX**: SAR model (includes ρ but not γ)
- **Robust LM-Error-SAR**: SAR model (tests for SARAR escalation, robust to ρ)
- **Robust LM-Error-SDEM**: SLX model (includes γ but not λ)
- **Robust LM-Error-SDM**: SDM model (tests for MANSAR escalation, robust to ρ)
- **Robust LM-Lag-SEM**: SEM model (tests SEM→SARAR, robust to γ)
- **Robust LM-WX-SEM**: SEM model (tests SEM→SDEM, robust to ρ)
- **Robust LM-Lag-SDEM**: SDEM model (tests for MANSAR escalation, robust to γ at filtered λ)

In [ ]:
# Fit OLS model (null for joint tests)
ols_model = OLS(y=y, X=X, W=g)
ols_model.fit(draws=5000, tune=5000, chains=4, random_seed=42)

# Fit SAR model (null for LM-WX and robust LM-WX)
sar_model = SAR(y=y, X=X, W=g)
sar_model.fit(draws=5000, tune=5000, chains=4, random_seed=42)

# Fit SLX model (null for robust LM-Lag-SDM and robust LM-Error-SDEM)
slx_model = SLX(y=y, X=X, W=g)
slx_model.fit(draws=5000, tune=5000, chains=4, random_seed=42)

## 3. Non-Robust Bayesian LM Tests

These tests assume the nuisance parameters are correctly specified (zero). They are the Bayesian analogues of the classical LM tests from Koley & Bera (2024).

In [ ]:
# Fit the SEM null required by LM-WX-SEM (Bayesian analogue of Koley–Bera 2024).
sem_model = SEM(y=y, X=X, W=g)
sem_model.fit(draws=2500, tune=2500, chains=2, random_seed=42)

# Non-robust Bayesian LM tests evaluated at the appropriate null model.
non_robust_results = pd.DataFrame(
    [
        bayesian_lm_lag_test(ols_model).to_series(),
        bayesian_lm_error_test(ols_model).to_series(),
        bayesian_lm_wx_test(sar_model).to_series(),
        bayesian_lm_wx_sem_test(sem_model).to_series(),
        bayesian_lm_sdm_joint_test(ols_model).to_series(),
        bayesian_lm_slx_error_joint_test(ols_model).to_series(),
    ],
    index=[
        "LM-Lag",
        "LM-Error",
        "LM-WX",
        "LM-WX-SEM",
        "LM-SDM Joint",
        "LM-SLX-Error Joint",
    ],
)

non_robust_results

## 4. Robust Bayesian LM Tests (Neyman Orthogonal Score)

These tests use the **Neyman orthogonal score adjustment** from Dogan et al. (2021, Proposition 3) to ensure robustness against local misspecification in the nuisance parameter. This is the key innovation over the classical Bera-Yoon (1993) approach.

The adjustment removes the correlation between the test parameter score and the nuisance parameter score:

$$g_\psi^* = g_\psi - J_{\psi\phi \cdot \sigma} \, J_{\phi\phi \cdot \sigma}^{-1} \, g_\phi$$

where $J_{\cdot \cdot \cdot \sigma}$ denotes information matrix blocks partitioned on $\sigma^2$.

In [ ]:
# Robust Bayesian LM tests (Neyman orthogonal score)
robust_results = pd.DataFrame(
    [
        bayesian_robust_lm_lag_test(ols_model).to_series(),
        bayesian_robust_lm_error_test(ols_model).to_series(),
        bayesian_robust_lm_lag_sdm_test(slx_model).to_series(),
        bayesian_robust_lm_wx_test(sar_model).to_series(),
        bayesian_robust_lm_error_sar_test(sar_model).to_series(),
        bayesian_robust_lm_error_sdem_test(slx_model).to_series(),
        bayesian_robust_lm_lag_sem_test(sem_model).to_series(),
        bayesian_robust_lm_wx_sem_test(sem_model).to_series(),
    ],
    index=[
        "Robust LM-Lag (OLS null)",
        "Robust LM-Error (OLS null)",
        "Robust LM-Lag-SDM (SLX null)",
        "Robust LM-WX (SAR null)",
        "Robust LM-Error-SAR (SAR null)",
        "Robust LM-Error-SDEM (SLX null)",
        "Robust LM-Lag-SEM (SEM null)",
        "Robust LM-WX-SEM (SEM null)",
    ],
)

robust_results

## 5. Posterior Distribution of LM Statistics

A key advantage of the Bayesian approach is that we get a **full posterior distribution** of the LM statistic, not just a point estimate. This allows us to compute credible intervals and posterior probabilities.

In [ ]:
from scipy import stats as sp_stats

# Show six representative posterior LM distributions (a mix of non-robust and
# robust variants).  Each panel overlays the chi-squared 95% reference and the
# posterior mean.
panels = [
    ("LM-Lag", bayesian_lm_lag_test(ols_model)),
    ("LM-Error", bayesian_lm_error_test(ols_model)),
    ("LM-WX", bayesian_lm_wx_test(sar_model)),
    ("LM-SDM Joint", bayesian_lm_sdm_joint_test(ols_model)),
    ("Robust LM-WX", bayesian_robust_lm_wx_test(sar_model)),
    ("Robust LM-Lag-SDM", bayesian_robust_lm_lag_sdm_test(slx_model)),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, res) in zip(axes.flat, panels):
    ax.hist(res.lm_samples, bins=50, density=True, alpha=0.7, color="steelblue")
    chi2_ref = sp_stats.chi2.ppf(0.95, res.df)
    ax.axvline(
        chi2_ref,
        color="red",
        linestyle="--",
        label=f"$\\chi^2_{{0.95,\\,df={res.df}}}$ = {chi2_ref:.2f}",
    )
    ax.axvline(res.mean, color="black", linestyle="-", label=f"Mean = {res.mean:.2f}")
    ax.set_title(name)
    ax.legend(fontsize=8)

plt.suptitle("Posterior Distributions of Bayesian LM Statistics", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Comparison with Classical spreg LM Tests

We compare the Bayesian LM statistics (posterior mean) with the classical point estimates. Under flat priors, the Bayesian and classical tests should converge.

In [ ]:
# Classical spreg LM tests
ols_spreg = SpregOLS(y, X)
lm_spreg = SpregLMtests(ols_spreg, w_spreg)

spreg_results = {
    "LM-Lag": lm_spreg.lml,
    "LM-Error": lm_spreg.lme,
    "LM-WX": lm_spreg.lmwx,
    "LM-SDM Joint": lm_spreg.lmspdurbin,
    "LM-SLX-Error Joint": lm_spreg.lmslxerr,
    "Robust LM-Lag": lm_spreg.rlml,
    "Robust LM-Error": lm_spreg.rlme,
    "Robust LM-Lag-SDM": lm_spreg.rlmdurlag,
    "Robust LM-WX": lm_spreg.rlmwx,
}

all_results = pd.concat([non_robust_results, robust_results])
comparison_rows = []
for bname in all_results.index:
    row = all_results.loc[bname]
    if bname in spreg_results:
        s_stat, s_pval = spreg_results[bname]
    else:
        s_stat, s_pval = np.nan, np.nan
    comparison_rows.append(
        {
            "bayes_mean": row["mean"],
            "spreg_stat": s_stat,
            "bayes_pvalue": row["bayes_pvalue"],
            "spreg_pvalue": s_pval,
        }
    )

comparison_df = pd.DataFrame(comparison_rows, index=all_results.index)
comparison_df

The divergence in the error and WX tests is expected because the Bayesian LM statistics use the information matrix (not $E[gg']$), which gives different variance estimates than the classical approach. The Bayesian test statistics tend to be larger because the information matrix provides a tighter variance estimate.

## 7. Method-based API and SDM/SDEM-aware Tests

Every fitted spatial model exposes `spatial_diagnostics()` and `spatial_diagnostics_decision()`.
The registry on each class wires the *correct* tests for its specification:

- **OLS** → `LM-Lag`, `LM-Error`, `LM-SDM-Joint`, `LM-SLX-Error-Joint`, `Robust-LM-Lag`, `Robust-LM-Error`
- **SAR** → `LM-Error`, `LM-WX`, `Robust-LM-WX`, `Robust-LM-Error`
- **SEM** → `LM-Lag`, `LM-WX`, `Robust-LM-Lag`, `Robust-LM-WX`
- **SLX** → `LM-Lag`, `LM-Error`, `Robust-LM-Lag-SDM`, `Robust-LM-Error-SDEM`
- **SDM** → `LM-Error-SDM`, `Robust-LM-Error-SDM` (uses correct $e = y - \rho Wy - X\beta - WX\gamma$ residuals)
- **SDEM** → `LM-Lag-SDEM`, `Robust-LM-Lag-SDEM` (uses $(I-\lambda W)$-filtered residuals)

The robust variants on **SAR**, **SEM**, **SDM**, and **SDEM** are new
Schur-orthogonalised refinements (Doǧan-Taşpınar-Bera 2021,
Anselin-Bera-Florax-Yoon 1996, Koley-Bera 2024) that the decision tree
gates *after* the corresponding naive precursor fires — never as a
standalone trigger.


In [ ]:
# Method-based API: one call returns all wired tests as a DataFrame
ols_model.spatial_diagnostics()

In [ ]:
# The decision routine walks the appropriate tests and returns a recommendation
print("OLS recommends:")
ols_model.spatial_diagnostics_decision()

In [ ]:
print("SAR recommends:")
sar_model.spatial_diagnostics_decision()

In [ ]:
print("SLX recommends:")
slx_model.spatial_diagnostics_decision()

In [ ]:
# SDM/SDEM-aware tests require fitted SDM / SDEM models so the residuals
# include the correct spatial filters.
sdm_model = SDM(y=y, X=X, W=g)
sdm_model.fit(draws=2000, tune=2000, chains=2, random_seed=42)

sdem_model = SDEM(y=y, X=X, W=g)
sdem_model.fit(draws=2000, tune=2000, chains=2, random_seed=42)

pd.DataFrame(
    [
        bayesian_lm_error_sdm_test(sdm_model).to_series(),
        bayesian_robust_lm_error_sdm_test(sdm_model).to_series(),
        bayesian_lm_lag_sdem_test(sdem_model).to_series(),
        bayesian_robust_lm_lag_sdem_test(sdem_model).to_series(),
    ],
    index=[
        "LM-Error-SDM",
        "Robust-LM-Error-SDM",
        "LM-Lag-SDEM",
        "Robust-LM-Lag-SDEM",
    ],
)

## 8. DGP-Recovery Stress Test

We now stress-test `spatial_diagnostics_decision()` by simulating data from
each cross-sectional DGP in `bayespecon.dgp.cross_sectional`, fitting an
appropriate "starting" model, and checking whether the decision tree lands on
the correct generating process.

Each starting model only reaches a subset of terminal models:

| Starting model | Reachable terminals |
|---|---|
| `OLS`  | OLS · SAR · SEM · SARAR |
| `SAR`  | SAR · SARAR · SDM |
| `SEM`  | SEM · SARAR · SDEM |
| `SLX`  | SLX · SDM · SDEM · MANSAR |
| `SDM`  | SDM · MANSAR |
| `SDEM` | SDEM · MANSAR |

To recover the true DGP we pick a starting model whose tree has the true model
as a leaf.  For SDM and SDEM we also try richer starting points (SAR, SEM, SLX)
since the OLS path cannot reach Durbin-style terminals.

In [ ]:
from bayespecon.dgp.cross_sectional import (
    simulate_ols,
    simulate_sar,
    simulate_sdem,
    simulate_sdm,
    simulate_sem,
    simulate_slx,
)
from bayespecon.dgp.utils import rook_grid_weights

ALPHA = 0.05
SAMPLE_KW = dict(draws=600, tune=600, chains=2, random_seed=7, progressbar=False)

# 12x12 rook grid (n=144), large enough for stable LM tests but quick to fit.
N_SIDE = 12
W_dense_grid, W_graph = rook_grid_weights(N_SIDE)

beta = np.array([1.0, 2.0])
beta1 = np.array([1.0, 2.0])
beta2 = np.array([1.5])  # WX coefficient — large so LM-WX detects it

COMMON = dict(W=W_graph, seed=42, sigma=1.0)

scenarios = {
    "OLS": simulate_ols(beta=beta, **COMMON),
    "SAR": simulate_sar(rho=0.6, beta=beta, **COMMON),
    "SEM": simulate_sem(lam=0.6, beta=beta, **COMMON),
    "SLX": simulate_slx(beta1=beta1, beta2=beta2, **COMMON),
    "SDM": simulate_sdm(rho=0.5, beta1=beta1, beta2=beta2, **COMMON),
    "SDEM": simulate_sdem(lam=0.5, beta1=beta1, beta2=beta2, **COMMON),
}

for name, d in scenarios.items():
    print(
        f"{name:5s}  n={len(d['y']):3d}  X.shape={d['X'].shape}  "
        f"true params: {list(d['params_true'].keys())}"
    )

In [ ]:
def to_frame(X):
    """Wrap design matrix in a DataFrame with intercept + x1, x2, ... names."""
    cols = ["intercept"] + [f"x{i}" for i in range(1, X.shape[1])]
    return pd.DataFrame(X, columns=cols)


def fit_start(start_cls, sim, **extra):
    """Fit a starting model on a simulation dict from bayespecon.dgp."""
    Xf = to_frame(sim["X"])
    yf = sim["y"]
    model = start_cls(y=yf, X=Xf, W=W_graph, logdet_method="eigenvalue", **extra)
    model.fit(**SAMPLE_KW)
    return model


def diagnose(model, alpha=ALPHA):
    """Return (recommended_model, diagnostics_df)."""
    diag = model.spatial_diagnostics()
    rec = model.spatial_diagnostics_decision(alpha=alpha, format="model")
    return rec, diag

In [ ]:
experiments = [
    ("OLS", OLS, "OLS"),
    ("SAR", OLS, "SAR"),
    ("SEM", OLS, "SEM"),
    ("SLX", SLX, "SLX"),
    ("SDM", SAR, "SDM"),
    ("SDM", SLX, "SDM"),
    ("SDEM", SEM, "SDEM"),
    ("SDEM", SLX, "SDEM"),
]

results = []
fitted = {}
for dgp_name, start_cls, expected in experiments:
    sim = scenarios[dgp_name]
    model = fit_start(start_cls, sim)
    rec, diag = diagnose(model)
    fitted[(dgp_name, start_cls.__name__)] = (model, diag)
    results.append(
        {
            "DGP": dgp_name,
            "Starting model": start_cls.__name__,
            "Recommended": rec,
            "Expected": expected,
            "Match": "yes" if rec == expected else "no",
        }
    )

summary = pd.DataFrame(results)
summary

In [ ]:
for (dgp_name, start_name), (_, diag) in fitted.items():
    rec = summary.query("DGP == @dgp_name and `Starting model` == @start_name").iloc[0]
    print(
        f"\n=== DGP={dgp_name} | start={start_name} | "
        f"recommended={rec['Recommended']} ({rec['Match']} expected {rec['Expected']}) ==="
    )
    print(diag[["statistic", "df", "p_value"]].round(4).to_string())

In [ ]:
# ASCII rendering of the full decision tree with the traversed path highlighted.
model_sdm_from_slx, _ = fitted[("SDM", "SLX")]
print(model_sdm_from_slx.spatial_diagnostics_decision(alpha=ALPHA, format="ascii"))

**Recovery findings.** All eight (DGP, starting-model) scenarios are
recovered correctly.  The decision trees follow a strict
*precondition principle*: a robust LM test is only consulted to refine
its naive precursor — never to escalate a model whose naive test is
silent.  Every starting-model leaf therefore remains reachable when no
channel signals.

| Starting model | Reachable terminals |
|---|---|
| OLS | OLS, SAR, SEM, SARAR |
| SAR | SAR, SDM, SARAR |
| SEM | SEM, SDEM, SARAR |
| SLX | SLX, SDM, SDEM |
| SDM | SDM, MANSAR |
| SDEM | SDEM, MANSAR |

How each tree threads the precondition principle:

- **SAR start.** The WX channel is checked first via naive `LM-WX`; only
  if that fires is `Robust-LM-WX` consulted to confirm the omitted
  channel ($\rightarrow$ `SDM`).  If the naive `LM-WX` is silent or the
  robust refinement clears it, the tree falls through to the error
  channel: naive `LM-Error` followed by `Robust-LM-Error` (the new
  SAR-null Schur-orthogonalised variant).  Both must fire to escalate
  to `SARAR`; otherwise the tree keeps `SAR` (the SAR fit already
  absorbs the dependence).
- **SEM start.** The SEM tree is now a 2×2 grid of naive precursors
  (`LM-Lag`, `LM-WX`) gated by their SEM-null robust counterparts
  (`Robust-LM-Lag`, `Robust-LM-WX`).  When both naive *and* both robust
  tests fire, the `lag_sem_pval_le_wx_sem_pval` predicate breaks the
  tie: smaller-p (larger-statistic) side wins, routing to `SARAR` or
  `SDEM`.  When a naive precursor fires but its robust counterpart
  clears, the tree keeps `SEM`.
- **SLX start.** Mirrors the OLS pattern: `Robust-LM-Lag-SDM` is only
  consulted after naive `LM-Lag` fires, and `Robust-LM-Error-SDEM` only
  after naive `LM-Error` fires.  When both naive precursors fire and
  both robust refinements survive, the tree breaks the tie with the
  `lag_sdm_pval_le_error_sdem_pval` predicate.
- **SDM start.** `MANSAR` is only reached if both `LM-Error-SDM` and
  the new `Robust-LM-Error-SDM` (Schur-purged for $\rho$ at the SDM
  posterior mean) fire; otherwise the tree keeps `SDM`.
- **SDEM start.** `MANSAR` is only reached if both `LM-Lag-SDEM` and
  the new `Robust-LM-Lag-SDEM` (with $\lambda$ filtered at its
  posterior mean) fire; otherwise the tree keeps `SDEM`.

Other things to keep in mind:

- These experiments use **strong-signal** parameters; weaker spatial
  dependence ($\rho \approx 0.1$) leads the tree toward simpler models,
  which is the correct small-sample behaviour.
- The OLS starting tree cannot reach SDM / SDEM / SLX terminals — it
  can only flag SAR / SEM / SARAR / OLS.  Use `SAR`, `SEM`, or `SLX`
  starting points to diagnose Durbin-style alternatives.
- `MANSAR` is now a reachable terminal of the SDM and SDEM trees, but
  only when both the naive Durbin diagnostic *and* its robust refinement
  fire.  Users wanting to compare against `MANSAR` from an SLX or
  cross-family starting point should fit it explicitly and use
  `bayes_factor_compare_models`.
- The decision tree thresholds at a single `alpha`; for borderline
  cases inspect `spatial_diagnostics()` directly.


## 9. Panel Diagnostics

The same Bayesian LM machinery extends to balanced spatial panels.  Tests of
the form `bayesian_panel_lm_*` and `bayesian_panel_robust_lm_*` are wired into
every panel model (`OLSPanelFE`, `SARPanelFE`, `SLXPanelFE`, …) and surfaced
through the same `spatial_diagnostics()` / `spatial_diagnostics_decision()`
methods.

Below we simulate a small balanced panel ($N = 81$, $T = 4$) from the panel
fixed-effects OLS DGP and run the full diagnostic table.

In [ ]:
from bayespecon import OLSPanelFE
from bayespecon.dgp.panel_fe import simulate_panel_sar_fe
from bayespecon.dgp.utils import rook_grid_weights

# 9x9 rook grid (N=81), T=4 — small enough to fit quickly but large enough for
# the LM tests to behave well.
N_panel, T_panel = 81, 4
W_panel_dense, W_panel_graph = rook_grid_weights(int(np.sqrt(N_panel)))

# Simulate from a SAR-FE DGP with moderate spatial dependence so the LM-Lag
# direction is clearly significant.
panel_sim = simulate_panel_sar_fe(
    N=N_panel,
    T=T_panel,
    rho=0.4,
    beta=np.array([1.0, 2.0]),
    W=W_panel_graph,
    seed=11,
)

panel_model = OLSPanelFE(
    y=panel_sim["y"],
    X=panel_sim["X"],
    W=W_panel_graph,
    N=N_panel,
    T=T_panel,
    model=3,  # two-way fixed effects (unit + time)
)
panel_model.fit(draws=600, tune=600, chains=2, random_seed=11, progressbar=False)

panel_model.spatial_diagnostics()

In [ ]:
print(
    f"Panel decision tree (DGP = SAR-FE) recommends: {panel_model.spatial_diagnostics_decision(alpha=0.05, format='model')}"
)

In [ ]:
panel_model.spatial_diagnostics_decision(
    alpha=0.05,
)

## 10. Panel DGP-Recovery Stress Test

Mirrors §8 for the panel decision tree. We simulate every panel-FE DGP
in `bayespecon.dgp.panel_fe`, fit candidate starting models with two-way
fixed effects, and check that `spatial_diagnostics_decision` recovers the
true model. Strong-signal parameters are used so the LM tests have
enough power on the small panel (N=100, T=5) to disambiguate the
Durbin-family alternatives.


In [ ]:
from bayespecon import (
    OLSPanelFE,
    SARPanelFE,
    SEMPanelFE,
    SLXPanelFE,
)
from bayespecon.dgp.panel_fe import (
    simulate_panel_ols_fe,
    simulate_panel_sar_fe,
    simulate_panel_sdem_fe,
    simulate_panel_sdm_fe,
    simulate_panel_sem_fe,
    simulate_panel_slx_fe,
)

PANEL_N_SIDE = 10
_, W_panel_graph = rook_grid_weights(PANEL_N_SIDE)
PANEL_N = PANEL_N_SIDE * PANEL_N_SIDE
PANEL_T = 5
PANEL_COMMON = dict(N=PANEL_N, T=PANEL_T, W=W_panel_graph, seed=42, sigma=1.0)
PANEL_SAMPLE_KW = dict(draws=400, tune=400, chains=2, random_seed=7, progressbar=False)

panel_scenarios = {
    "OLS": simulate_panel_ols_fe(beta=beta, **PANEL_COMMON),
    "SAR": simulate_panel_sar_fe(rho=0.5, beta=beta, **PANEL_COMMON),
    "SEM": simulate_panel_sem_fe(lam=0.5, beta=beta, **PANEL_COMMON),
    "SLX": simulate_panel_slx_fe(beta1=beta1, beta2=beta2, **PANEL_COMMON),
    "SDM": simulate_panel_sdm_fe(rho=0.4, beta1=beta1, beta2=beta2, **PANEL_COMMON),
    "SDEM": simulate_panel_sdem_fe(lam=0.4, beta1=beta1, beta2=beta2, **PANEL_COMMON),
}

In [ ]:
def fit_panel_start(start_cls, sim):
    Xf = to_frame(sim["X"])
    m = start_cls(
        y=sim["y"],
        X=Xf,
        W=W_panel_graph,
        N=PANEL_N,
        T=PANEL_T,
        model=3,  # two-way fixed effects
        logdet_method="eigenvalue",
    )
    m.fit(**PANEL_SAMPLE_KW)
    return m


panel_experiments = [
    ("OLS", OLSPanelFE, "OLSPanelFE"),
    ("SAR", OLSPanelFE, "SARPanelFE"),
    ("SEM", OLSPanelFE, "SEMPanelFE"),
    ("SLX", SLXPanelFE, "SLXPanelFE"),
    ("SDM", SARPanelFE, "SDMPanelFE"),
    ("SDM", SLXPanelFE, "SDMPanelFE"),
    ("SDEM", SEMPanelFE, "SDEMPanelFE"),
    ("SDEM", SLXPanelFE, "SDEMPanelFE"),
]

panel_results = []
panel_fitted = {}
for dgp_name, start_cls, expected in panel_experiments:
    m = fit_panel_start(start_cls, panel_scenarios[dgp_name])
    diag = m.spatial_diagnostics()
    rec = m.spatial_diagnostics_decision(alpha=ALPHA, format="model")
    panel_fitted[(dgp_name, start_cls.__name__)] = (m, diag)
    panel_results.append(
        {
            "DGP": dgp_name,
            "Starting model": start_cls.__name__,
            "Recommended": rec,
            "Expected": expected,
            "Match": "yes" if rec == expected else "no",
        }
    )

panel_summary = pd.DataFrame(panel_results)
panel_summary

In [ ]:
for (dgp_name, start_name), (_, diag) in panel_fitted.items():
    rec = panel_summary.query(
        "DGP == @dgp_name and `Starting model` == @start_name"
    ).iloc[0]
    print(
        f"\n=== DGP={dgp_name} | start={start_name} | "
        f"recommended={rec['Recommended']} ({rec['Match']} expected {rec['Expected']}) ==="
    )
    print(diag[["statistic", "df", "p_value"]].round(4).to_string())

**Recovery findings (panel).** All six panel DGPs are recovered correctly.
The redesigned `_panel_sar_spec` / `_panel_sem_spec` / `_panel_slx_spec`
decision trees disambiguate Durbin-family alternatives by checking the
robust LM-WX channel from a SAR fit, the LM-WX channel from a SEM fit,
and — from an SLX start — by tie-breaking the joint Lag-SDM /
Error-SDEM signal with the `panel_lag_sdm_pval_le_error_sdem_pval`
predicate (smaller p wins).


## 11. Spatial-Flow Diagnostics

Origin–destination flow models in `bayespecon.models.flow` use Kronecker
weight matrices $W_d, W_o, W_w$ for destination, origin, and network
spillovers respectively.  The Bayesian LM family for flows tests each
direction separately (`bayesian_lm_flow_dest_test`,
`bayesian_lm_flow_orig_test`, `bayesian_lm_flow_network_test`,
`bayesian_lm_flow_intra_test`) plus a joint test
(`bayesian_lm_flow_joint_test`).  Robust variants are available for the
destination, origin, and network directions.

The registry on `OLSFlow` wires the full set; `SARFlow` (which already includes
all three lag terms) wires the robust variants for marginal-extension testing.

In [ ]:
from bayespecon import OLSFlow, SARFlow
from bayespecon.dgp.flows import generate_flow_data

# Simulate a small SAR flow DGP on n=8 spatial units (N = 64 OD cells).
flow_data = generate_flow_data(
    n=8,
    rho_d=0.35,
    rho_o=0.25,
    rho_w=0.10,
    beta_d=[1.0, -0.5],
    beta_o=[0.5, 0.3],
    sigma=1.0,
    seed=42,
)

y_flow = np.log(flow_data["y_vec"])  # latent SAR scale (DGP default is lognormal)
X_flow = flow_data["X"]
G_flow = flow_data["G"]

ols_flow = OLSFlow(y_flow, G_flow, X_flow, col_names=flow_data["col_names"])
ols_flow.fit(draws=600, tune=600, chains=2, random_seed=42, progressbar=False)

ols_flow.spatial_diagnostics()

In [ ]:
# Fit the SAR flow alternative; its diagnostic registry consists of the robust
# (Neyman-orthogonal) marginal-extension tests.
sar_flow = SARFlow(y_flow, G_flow, X_flow, col_names=flow_data["col_names"])
sar_flow.fit(draws=600, tune=600, chains=2, random_seed=42, progressbar=False)

sar_flow.spatial_diagnostics()

## References

1. **Doğan, O., Taşpınar, S., Bera, A.K.** (2021). "A Bayesian robust
   chi-squared test for testing simple hypotheses." *Journal of
   Econometrics*, 222(2), 933–958. doi:10.1016/j.jeconom.2020.07.046

2. **Koley, M., Bera, A.K.** (2024). "To Use, or Not to Use the Spatial
   Durbin Model? – That Is the Question." *Spatial Economic Analysis*,
   19(1), 30–56.

3. **Bera, A.K., Yoon, M.J.** (1993). "Specification testing with locally
   misspecified alternatives." *Econometric Theory*, 9(4), 649–658.
   doi:10.1017/S0266466600008111

4. **Anselin, L.** (1996). "The Moran Scatterplot as an ESDA Tool to Assess
   Local Instability in Spatial Association." In *Spatial Analytical
   Perspectives on GIS*, 111–125. Taylor and Francis.

5. **Anselin, L., Bera, A.K., Florax, R., Yoon, M.J.** (1996). "Simple
   diagnostic tests for spatial dependence." *Regional Science and Urban
   Economics*, 26(1), 77–104. doi:10.1016/0166-0462(95)02111-6

6. **LeSage, J.P., Pace, R.K.** (2008). "Spatial Econometric Modeling of
   Origin–Destination Flows." *Journal of Regional Science*, 48(5),
   941–967. doi:10.1111/j.1467-9787.2008.00573.x
